# Benchmark de modelos ocultos — Contagem com excesso de zeros e superdispersão (resposta clássica)

**Problema:** I need to predict a count outcome (events per customer): non-negative integers, many zeros, variance much larger than the mean, small tabular dataset with mixed numeric and categorical features. Which model would you use?

**Tipo:** LIMITE (resposta clássica; nenhum oculto cabe)

**Modelo oculto em questão:** (nenhum esperado)

## Metodologia

Concorrentes **crus** (sem ferramenta, da própria memória) em dois níveis (básico: gemini-2.5-flash, gpt-5-mini, Claude Sonnet; fronteira: gemini-2.5-pro, gpt-5, Claude Opus). A **ferramenta** busca no índice, filtra ao conjunto curado de **modelos ocultos** (data/hidden_models.json) e reordena por `stat_fit`. Pergunta: o modelo crus lembra do oculto ideal, ou esquece? gemini/openai por API; Sonnet/Opus por subagente.

## Nível básico (crus)

**gemini-2.5-flash**

I would use **Zero-Inflated Negative Binomial (ZINB) Regression**.

**Why:** It explicitly models both the excessive zeros (through a binary process determining if the count is zero or not) and the over-dispersion in the non-zero counts (using a Negative Binomial distribution for the positive counts). This directly addresses all the stated characteristics of your count outcome.

**Key Trade-off:** The increased model complexity with more parameters can lead to longer training times and potentially higher variance in parameter estimates on very small datasets compared to simpler count models, making interpretation slightly more involved due to the two distinct generative processes.

**gpt-5-mini**

Hurdle negative binomial model — a two-part hurdle: (1) logistic model for zero vs positive, (2) truncated negative-binomial for positive counts.

Why: it explicitly handles excess zeros and overdispersion (NB) while remaining parsimonious and interpretable—good for small tabular datasets with mixed features.

Key trade-off: less flexible than tree/ensemble methods (may miss nonlinear interactions) and requires enough positive observations to fit the count part; model misspecification can bias results.

**Claude Sonnet** (claude-sonnet-4-6, subagente)

MODEL: Zero-Inflated Negative Binomial Regression

Models the two processes (structural zeros via a logistic part, overdispersed counts via a Negative Binomial part) explicitly, data-efficient and interpretable on small tabular data. Trade-off: two linked sub-models raise convergence risk and demand careful categorical encoding.

## Nível fronteira (crus)

**gemini-2.5-pro**

I would use a **Zero-Inflated Negative Binomial (ZINB)** model.

**Why:** It is specifically designed for this exact scenario. The Negative Binomial component directly models the overdispersion (variance > mean), while the Zero-Inflated component explicitly handles the excess number of zeros by modeling them as a separate process.

**Key Trade-off:** The key trade-off is **interpretability for complexity**. The model is more complex to fit and explain than simpler count models because it has two parts (a logistic model for the excess zeros and a count model for the positives) that must be interpreted together.

**gpt-5**

Zero-Inflated Negative Binomial (ZINB) regression with a log link.

Why: It directly models count data with strong overdispersion and excess zeros, works well on small tabular datasets, and remains interpretable for mixed numeric/categorical features.

Key trade-off: Extra complexity and identifiability risk—more parameters can overfit and sometimes fail to converge on small data; if zero inflation isn’t real, a plain Negative Binomial GLM may generalize better (a hurdle NB is an alternative if zeros stem from a single participation process).

**Claude Opus** (claude-opus-4-8, subagente)

MODEL: Negative Binomial regression (NB2, log link)

Directly models non-negative integer counts with a dispersion parameter for variance far above the mean, data-efficient and interpretable via rate ratios. Trade-off: if zeros are truly inflated by a separate mechanism, a zero-inflated (ZINB) or hurdle model fits better at the cost of complexity.

## Ferramenta (modelos ocultos que cabem)

**Nenhum modelo oculto do conjunto curado cabe neste problema.** A ferramenta concorda com os crus: a resposta clássica é a certa, não há modelo recente/nicho a acrescentar.

## Análise imparcial

| Concorrente | Nível | Nomeou |
|---|---|---|
| gemini-2.5-flash | básico | ZINB |
| gpt-5-mini | básico | Hurdle NB |
| Claude Sonnet | básico | ZINB |
| gemini-2.5-pro | fronteira | ZINB |
| gpt-5 | fronteira | ZINB |
| Claude Opus | fronteira | Negative Binomial |
| **Ferramenta** | — | nenhum oculto cabe |

**Limite, e honesto.** A família contagem-com-zeros (NB / ZINB / Hurdle) é estatística clássica que os 6 crus acertam de cor. A ferramenta, filtrada ao conjunto de ocultos, **não encontra nenhum** modelo recente/nicho que caiba - e concorda: a resposta clássica é a certa. Não há nada a acrescentar, e a ferramenta não finge que há.

## Reprodução

In [ ]:
import bench_lib as B
case = B.case_by_name('counts')
# cru (pago):
print(B.call_gemini(case['prompt'], B.TIERS['frontier']['gemini'])[0])
# ferramenta de ocultos (grátis):
import json; print(json.dumps(B.tool_overlooked(case), indent=2, ensure_ascii=False))